# Personal-rating prediction

Compare a training-mean rating predictor with regularized linear regression (Ridge). The target is the numerical personal rating among completed dramas; this is distinct from recommendation ranking and enjoyment-probability estimation.

## Inputs and scope

Use the project environment with scikit-learn installed and a local enriched history snapshot. Copy this notebook to `notebooks/local/` before execution. Public copies contain no saved personal results.

The experiment excludes incomplete and unrated records. Features are genre indicators and public rating. Cast, year, episode count, notes and post-watch preference flags are not inputs. Missing public ratings use the training median; an all-missing training column uses 5.5. Only public rating is standardized; genre columns remain binary. Missing/unknown genres contribute no active known indicators.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
os.environ.setdefault('MPLCONFIGDIR', str(root / 'data/processed/.matplotlib'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display, Markdown
from dr_drama.analysis.eda import load_history
from dr_drama.models.rating_prediction import (
    ALPHAS, completed_history, evaluate_rating_model, fit_rating_model, predict_unwatched,
)
source = root / 'data/processed/watch_history_enriched.csv'
if not source.exists():
    raise FileNotFoundError('Prepare the enriched local history using the repository README.')
source_hash = hashlib.sha256(source.read_bytes()).hexdigest()
history = load_history(source)
completed = completed_history(history)
OUTER_FOLDS, INNER_FOLDS, SEED = 5, 3, 42
display(history.status.value_counts().rename('records').to_frame())
display(completed[['my_rating', 'public_rating']].describe().round(3))
display(Markdown(f'Eligible completed ratings: **{len(completed)}**. Missing public ratings: **{completed.public_rating.isna().sum()}**.'))
if 'mdl_match_status' in completed:
    display(completed.mdl_match_status.value_counts().rename('identity_status').to_frame())
plt.rcParams.update({'figure.figsize': (9, 4.5), 'axes.spines.top': False, 'axes.spines.right': False})


## Model and validation protocol

Ridge minimizes squared prediction error plus `alpha × sum(coefficient²)`, with an unpenalized intercept. Larger alpha applies stronger shrinkage. The alternative baseline predicts the training mean for every test item.

Five shuffled outer KFold splits (seed 42) provide out-of-fold predictions. Within each outer training set, three shuffled inner folds select alpha from `[0.1, 1, 10, 100]` by MAE. Vocabulary, imputation and scaling are refitted inside every pipeline fit. No outer labels select alpha. The mean baseline uses exactly the same outer training/test partitions.

Report pooled out-of-fold MAE as the primary metric and RMSE as a secondary measure sensitive to larger errors, alongside individual folds and inner search results. Tuning and evaluation use raw, unclipped predictions. Do not compare rating-point errors numerically with earlier Precision@5 ranking metrics.


In [ ]:
fold_metrics, predictions, tuning = evaluate_rating_model(
    history, outer_folds=OUTER_FOLDS, inner_folds=INNER_FOLDS, seed=SEED, alphas=ALPHAS)
summary_rows = []
for name in ['mean', 'ridge']:
    residual = predictions.actual_rating - predictions[name + '_prediction']
    summary_rows.append({'model': name, 'pooled_mae': residual.abs().mean(),
                        'pooled_rmse': np.sqrt(residual.pow(2).mean()),
                        'out_of_range_predictions': int((~predictions[name + '_prediction'].between(1, 10)).sum())})
summary = pd.DataFrame(summary_rows).set_index('model')
display(summary.round(3))
display(fold_metrics.round(3))
display(tuning.pivot(index='outer_fold', columns='alpha', values='inner_mean_mae').round(3))
improvement = summary.loc['mean', 'pooled_mae'] - summary.loc['ridge', 'pooled_mae']
direction = 'lower' if improvement > 0 else 'higher' if improvement < 0 else 'equal'
display(Markdown(f'Ridge pooled MAE is **{abs(improvement):.3f} points {direction}** than the mean baseline. This is a retrospective development result, not a probability, classification accuracy or prospective guarantee.'))


## Error diagnostics

The diagonal marks perfect predictions. Residuals are actual minus predicted rating: positive values indicate underprediction. Largest errors are descriptive diagnostics, not grounds to remove records. Changes informed by these errors require a new validation plan. Fold variation is not a confidence interval.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)
for ax, name in zip(axes, ['mean', 'ridge']):
    ax.scatter(predictions.actual_rating, predictions[name + '_prediction'], alpha=.65, color='#416d9c')
    ax.plot([1,10], [1,10], '--', color='#888888', linewidth=1)
    ax.set(xlabel='Actual personal rating', ylabel='Held-out predicted rating', title=name.title())
fig.tight_layout(); display(fig); plt.close(fig)
errors = predictions.assign(ridge_residual=predictions.actual_rating - predictions.ridge_prediction)
display(errors.sort_values('ridge_absolute_error', ascending=False)[
    ['canonical_title', 'actual_rating', 'ridge_prediction', 'ridge_residual', 'ridge_absolute_error', 'mean_absolute_error']].head(10).round(2))
fig, ax = plt.subplots()
ax.scatter(errors.ridge_prediction, errors.ridge_residual, alpha=.65, color='#416d9c')
ax.axhline(0, color='#888888', linestyle='--')
ax.set(xlabel='Held-out Ridge prediction', ylabel='Actual minus predicted rating', title='Held-out residuals')
fig.tight_layout(); display(fig); plt.close(fig)


## Full-history refit and candidate estimates

After evaluation, inner cross-validation chooses alpha on all eligible history for a separate refit. Its training predictions are not used as performance evidence. Coefficients describe conditional model associations, not causal effects. Genre weights correspond to binary changes; the public-score weight corresponds to one training standard deviation. Correlated genres and regularization limit interpretation.

When a candidate catalog is available, all recorded IDs are excluded and remaining candidates receive estimated ratings. Raw predictions remain visible; display-only clipping to 1–10 is flagged. Unknown genres and imputed public scores are reported. These estimates are conditional on completion and do not predict whether the user will drop a drama.


In [ ]:
final_search = fit_rating_model(history, alphas=ALPHAS, inner_folds=INNER_FOLDS, seed=SEED)
estimator = final_search.best_estimator_
feature_transform = estimator.named_steps['features']
ridge = estimator.named_steps['ridge']
coefficients = pd.DataFrame({'feature': feature_transform.get_feature_names_out(), 'coefficient': ridge.coef_})
display(coefficients.sort_values('coefficient', ascending=False).round(3))
display(Markdown(f"Full-history selected alpha: **{final_search.best_params_['ridge__alpha']}**; intercept: **{ridge.intercept_:.3f}**. These are fitted parameters, not held-out performance."))
catalog_path = root / 'data/processed/candidate_catalog.csv'
candidate_predictions = pd.DataFrame()
catalog_hash = None
if catalog_path.exists():
    catalog_hash = hashlib.sha256(catalog_path.read_bytes()).hexdigest()
    catalog = pd.read_csv(catalog_path, dtype=str, keep_default_na=False)
    candidate_predictions = predict_unwatched(final_search, catalog, history)
    display(candidate_predictions.round(2))
else:
    print('No local candidate catalog; held-out evaluation is complete without live examples.')
assert hashlib.sha256(source.read_bytes()).hexdigest() == source_hash


## Limitations

This is an exploratory single-user experiment. Previously inspected history is not a fresh final test; random folds do not reconstruct viewing order or historical public scores. Provisional identities, franchise overlap, selected viewing habits and sparse attributes limit generalization. Missing genres can be indistinguishable from no known memberships. No interaction terms, formal uncertainty intervals or probability calibration are implemented.

Alpha selection is limited to a predefined grid. A boundary selection is a limitation to report, not a reason to expand the grid against outer test outcomes. Ridge optimizes squared error while alpha selection uses MAE. A median baseline is a possible future comparator aligned with absolute loss; the current mean comparator follows the initial experiment definition.

No agent or deployment is included. Further development requires fresh validation and prospective feedback.

References: [Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html), [nested cross-validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html).
